# Representación del estado y capas $(\rho)$ y $(\pi)$ de Keccak

Este notebook implementa y valida las primeras transformaciones estructurales del modelo reducido de Keccak.

El estado se representa como:

$[
A[x,y,k],
]$

donde:

$[
x,y\in\{0,1,2,3,4\},
\qquad
k\in\{0,\ldots,z-1\}.
]$

Por tanto, el estado tiene dimensiones:

$[
5\times5\times z.
]$

Para los valores considerados en el trabajo:

$[
z=4 \Rightarrow 100\text{ bits},
]$

$[
z=8 \Rightarrow 200\text{ bits}.
]$

En esta etapa se estudian:

- $(\rho)$: rotación de los bits dentro de cada *lane*;
- $(\pi)$: permutación de las posiciones de los *lanes*.

Ambas transformaciones son lineales y no cambian el número total de bits activos.

## 1. Configuración del proyecto

El código fuente se encuentra en la carpeta `src`. Se incorpora dicha ruta al entorno del notebook para importar el paquete local `keccak_milp`.

In [1]:
# ============================================================
# CONFIGURACIÓN DE RUTAS
# ============================================================

from pathlib import Path
import sys


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


print(f"Directorio actual : {CURRENT_DIR}")
print(f"Raíz del proyecto : {PROJECT_ROOT}")
print(f"Carpeta src       : {SRC_DIR}")
print(f"src existe        : {SRC_DIR.exists()}")

Directorio actual : d:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\notebooks
Raíz del proyecto : d:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada
Carpeta src       : d:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\src
src existe        : True


## 2. Importación de las funciones

Se importan las funciones encargadas de:

- crear estados etiquetados;
- obtener los desplazamientos de $(\rho)$;
- aplicar $(\rho$);
- aplicar $(\pi)$;
- aplicar la composición $(\rho\circ\pi)$;
- comprobar que las transformaciones sean permutaciones.

In [2]:
# ============================================================
# IMPORTACIONES
# ============================================================

import numpy as np
import pandas as pd

from keccak_milp.layers import (
    RHO_OFFSETS,
    create_labeled_state,
    is_permutation,
    pi,
    pi_destination,
    rho,
    rho_offset,
    rho_pi,
    rho_pi_destination,
)


print("Funciones importadas correctamente.")

Funciones importadas correctamente.


## 3. Representación del estado de Keccak

Para facilitar la validación, cada posición del estado se etiqueta con un número único.

Por ejemplo, para $(z=4)$, el estado contiene:

$[
25(4)=100
]$

posiciones diferentes, numeradas entre 0 y 99.

Estos valores no representan todavía bits criptográficos. Son identificadores que permiten seguir el movimiento de cada posición después de aplicar una transformación.

In [3]:
# ============================================================
# CREACIÓN DE ESTADOS ETIQUETADOS
# ============================================================

estado_z4 = create_labeled_state(z=4)
estado_z8 = create_labeled_state(z=8)


print("=" * 60)
print("ESTADO z = 4")
print("=" * 60)
print(f"Forma       : {estado_z4.shape}")
print(f"Elementos   : {estado_z4.size}")
print(f"Valores únicos: {len(np.unique(estado_z4))}")

print()

print("=" * 60)
print("ESTADO z = 8")
print("=" * 60)
print(f"Forma       : {estado_z8.shape}")
print(f"Elementos   : {estado_z8.size}")
print(f"Valores únicos: {len(np.unique(estado_z8))}")

ESTADO z = 4
Forma       : (5, 5, 4)
Elementos   : 100
Valores únicos: 100

ESTADO z = 8
Forma       : (5, 5, 8)
Elementos   : 200
Valores únicos: 200


## 4. Visualización de los *lanes*

Cada posición $((x,y))$ identifica un *lane*. Para $(z=4)$, cada *lane* contiene cuatro posiciones:

$[
A[x,y,:].
]$

La siguiente tabla muestra los 25 *lanes* del estado inicial.

In [4]:
# ============================================================
# TABLA DE LANES PARA z = 4
# ============================================================

registros_lanes = []

for x in range(5):
    for y in range(5):
        registros_lanes.append(
            {
                "x": x,
                "y": y,
                "lane": estado_z4[x, y, :].tolist(),
            }
        )

df_lanes_z4 = pd.DataFrame(registros_lanes)

df_lanes_z4

,x,y,lane
0,0,0,"[0, 1, 2, 3]"
1,0,1,"[4, 5, 6, 7]"
2,0,2,"[8, 9, 10, 11]"
3,0,3,"[12, 13, 14, 15]"
4,0,4,"[16, 17, 18, 19]"
5,1,0,"[20, 21, 22, 23]"
6,1,1,"[24, 25, 26, 27]"
7,1,2,"[28, 29, 30, 31]"
8,1,3,"[32, 33, 34, 35]"
9,1,4,"[36, 37, 38, 39]"


## 5. Transformación $(\rho)$

La transformación $(\rho)$ rota cada *lane* una cantidad predeterminada de posiciones.

Formalmente:

$$[
A_{\rho}[x,y,(k+r[x,y])\bmod z]
=
A[x,y,k],
]$$

donde $(r[x,y])$ es el desplazamiento definido por Keccak.

En las versiones reducidas se utiliza:

$$[
r_z[x,y]=r[x,y]\bmod z.
]$$

Por tanto, los desplazamientos efectivos dependen del valor de $(z)$.

In [5]:
# ============================================================
# TABLA DE DESPLAZAMIENTOS RHO
# ============================================================

tabla_rho = []

for x in range(5):
    for y in range(5):
        tabla_rho.append(
            {
                "x": x,
                "y": y,
                "offset_oficial": RHO_OFFSETS[x][y],
                "offset_z4": rho_offset(x, y, 4),
                "offset_z8": rho_offset(x, y, 8),
            }
        )

df_rho_offsets = pd.DataFrame(tabla_rho)

df_rho_offsets

,x,y,offset_oficial,offset_z4,offset_z8
0,0,0,0,0,0
1,0,1,36,0,4
2,0,2,3,3,3
3,0,3,41,1,1
4,0,4,18,2,2
5,1,0,1,1,1
6,1,1,44,0,4
7,1,2,10,2,2
8,1,3,45,1,5
9,1,4,2,2,2


In [6]:
# ============================================================
# APLICACIÓN DE RHO
# ============================================================

estado_rho_z4 = rho(estado_z4)
estado_rho_z8 = rho(estado_z8)


print("Rho para z=4 es permutación:",
      is_permutation(estado_z4, estado_rho_z4))

print("Rho para z=8 es permutación:",
      is_permutation(estado_z8, estado_rho_z8))

Rho para z=4 es permutación: True
Rho para z=8 es permutación: True


### Ejemplo de rotación

Para el *lane*:

$[
A[1,0,:],
]$

el desplazamiento oficial de $(\rho)$ es 1.

Si:

$[
A[1,0,:]=[a_0,a_1,a_2,a_3],
]$

entonces, para $(z=4)$:

$$[
$\rho$(A[1,0,:])
=
[a_3,a_0,a_1,a_2].
]$

In [7]:
# ============================================================
# EJEMPLO DE ROTACIÓN DE UN LANE
# ============================================================

x_ejemplo = 1
y_ejemplo = 0

lane_antes = estado_z4[x_ejemplo, y_ejemplo, :]
lane_despues = estado_rho_z4[x_ejemplo, y_ejemplo, :]

print(f"Coordenada lane : ({x_ejemplo}, {y_ejemplo})")
print(f"Desplazamiento  : {rho_offset(x_ejemplo, y_ejemplo, 4)}")
print(f"Antes de rho    : {lane_antes}")
print(f"Después de rho  : {lane_despues}")

Coordenada lane : (1, 0)
Desplazamiento  : 1
Antes de rho    : [20 21 22 23]
Después de rho  : [23 20 21 22]


## 6. Transformación $(\pi)$

La transformación $(\pi)$ cambia la posición de los *lanes*.

Se utiliza la relación:

$$[
B[y,(2x+3y)\bmod5]
=
A[x,y].
]$$

La transformación no altera los bits dentro de cada *lane*. Únicamente mueve el *lane* completo a otra coordenada.

In [8]:
# ============================================================
# MAPEO DE COORDENADAS DE PI
# ============================================================

mapeo_pi = []

for x in range(5):
    for y in range(5):
        x_destino, y_destino = pi_destination(x, y)

        mapeo_pi.append(
            {
                "x_origen": x,
                "y_origen": y,
                "x_destino": x_destino,
                "y_destino": y_destino,
            }
        )

df_mapeo_pi = pd.DataFrame(mapeo_pi)

df_mapeo_pi

,x_origen,y_origen,x_destino,y_destino
0,0,0,0,0
1,0,1,1,3
2,0,2,2,1
3,0,3,3,4
4,0,4,4,2
5,1,0,0,2
6,1,1,1,0
7,1,2,2,3
8,1,3,3,1
9,1,4,4,4


In [9]:
# ============================================================
# APLICACIÓN DE PI
# ============================================================

estado_pi_z4 = pi(estado_z4)
estado_pi_z8 = pi(estado_z8)


print("Pi para z=4 es permutación:",
      is_permutation(estado_z4, estado_pi_z4))

print("Pi para z=8 es permutación:",
      is_permutation(estado_z8, estado_pi_z8))

Pi para z=4 es permutación: True
Pi para z=8 es permutación: True


### Ejemplo de permutación de un *lane*

El *lane* ubicado en:

$[
A[1,0]
]$

se desplaza según:

$[
x'=y=0,
]$

$[
y'=(2x+3y)\bmod5=2.
]$

Por tanto:

$[
A[1,0]\longrightarrow B[0,2].
]$

In [10]:
# ============================================================
# EJEMPLO DE MOVIMIENTO DE UN LANE
# ============================================================

x_origen = 1
y_origen = 0

x_destino, y_destino = pi_destination(
    x_origen,
    y_origen,
)

lane_origen = estado_z4[x_origen, y_origen, :]
lane_destino = estado_pi_z4[x_destino, y_destino, :]


print(f"Origen        : A[{x_origen}, {y_origen}]")
print(f"Destino       : B[{x_destino}, {y_destino}]")
print(f"Lane origen   : {lane_origen}")
print(f"Lane destino  : {lane_destino}")

assert np.array_equal(
    lane_origen,
    lane_destino,
)

print("El lane fue trasladado correctamente.")

Origen        : A[1, 0]
Destino       : B[0, 2]
Lane origen   : [20 21 22 23]
Lane destino  : [20 21 22 23]
El lane fue trasladado correctamente.


## 7. Composición $(\rho)$ y $(\pi)$

Dentro de una ronda de Keccak, primero se aplica $(\rho)$ y después $(\pi)$:

$[
A
\xrightarrow{\rho}
A_{\rho}
\xrightarrow{\pi}
B.
]$

Para un bit ubicado originalmente en:

$[
A[x,y,k],
]$

su posición después de ambas transformaciones es:

$[
x'=y,
]$

$[
y'=(2x+3y)\bmod5,
]$

$[
k'=(k+r[x,y])\bmod z.
]$

Esta relación será utilizada posteriormente para construir las restricciones o equivalencias del modelo MILP.

In [11]:
# ============================================================
# COMPOSICIÓN RHO + PI
# ============================================================

estado_rho_pi_z4 = rho_pi(estado_z4)
estado_rho_pi_z8 = rho_pi(estado_z8)


print("Rho + Pi para z=4 es permutación:",
      is_permutation(estado_z4, estado_rho_pi_z4))

print("Rho + Pi para z=8 es permutación:",
      is_permutation(estado_z8, estado_rho_pi_z8))

Rho + Pi para z=4 es permutación: True
Rho + Pi para z=8 es permutación: True


In [12]:
# ============================================================
# SEGUIMIENTO DE UN BIT
# ============================================================

x_origen = 1
y_origen = 0
k_origen = 2
z = 4

destino = rho_pi_destination(
    x=x_origen,
    y=y_origen,
    k=k_origen,
    z=z,
)

valor_origen = estado_z4[
    x_origen,
    y_origen,
    k_origen,
]

valor_destino = estado_rho_pi_z4[destino]


print(f"Coordenada origen : ({x_origen}, {y_origen}, {k_origen})")
print(f"Coordenada destino: {destino}")
print(f"Valor origen      : {valor_origen}")
print(f"Valor destino     : {valor_destino}")

assert valor_origen == valor_destino

print("El seguimiento del bit fue validado correctamente.")

Coordenada origen : (1, 0, 2)
Coordenada destino: (0, 2, 3)
Valor origen      : 22
Valor destino     : 22
El seguimiento del bit fue validado correctamente.


## 8. Validaciones globales

Se comprobarán las siguientes propiedades:

1. el estado conserva su forma;
2. $(\rho)$ conserva todos los elementos;
3. $(\pi)$ conserva todos los elementos;
4. la composición $(\rho+\pi)$ conserva todos los elementos;
5. cada posición de origen tiene exactamente una posición de destino;
6. las propiedades se cumplen para $(z=4)$ y $(z=8)$.

In [13]:
# ============================================================
# VALIDACIONES GLOBALES
# ============================================================

for z in (4, 8):
    estado = create_labeled_state(z)

    estado_rho = rho(estado)
    estado_pi = pi(estado)
    estado_rho_pi = rho_pi(estado)

    assert estado.shape == (5, 5, z)
    assert estado_rho.shape == estado.shape
    assert estado_pi.shape == estado.shape
    assert estado_rho_pi.shape == estado.shape

    assert is_permutation(estado, estado_rho)
    assert is_permutation(estado, estado_pi)
    assert is_permutation(estado, estado_rho_pi)

    destinos = set()

    for x in range(5):
        for y in range(5):
            for k in range(z):
                destino = rho_pi_destination(
                    x=x,
                    y=y,
                    k=k,
                    z=z,
                )

                destinos.add(destino)

                assert estado_rho_pi[destino] == estado[x, y, k]

    assert len(destinos) == 25 * z

    print(
        f"Validación completada para z={z}: "
        f"{len(destinos)} posiciones únicas."
    )

Validación completada para z=4: 100 posiciones únicas.
Validación completada para z=8: 200 posiciones únicas.


## 9. Interpretación para el modelo MILP

Las transformaciones $(\rho)$ y $(\pi)$ no requieren una linealización especial.

Si una variable binaria representa la actividad de una posición antes de estas capas:

$[
a_{x,y,k},
]$

la variable correspondiente después de $(\rho)$ y $(\pi)$ puede definirse mediante una igualdad:

$$[
b_{y,(2x+3y)\bmod5,(k+r[x,y])\bmod z}
=
a_{x,y,k}.
]$$

También puede utilizarse directamente una correspondencia de índices, evitando crear variables auxiliares innecesarias.

Esto reduce:

- el número de variables;
- el número de restricciones;
- el tiempo de resolución;
- el consumo de memoria.

La siguiente etapa incorporará la transformación $(\theta)$, que utiliza operaciones XOR y requiere una representación MILP de paridad.

In [14]:
# ============================================================
# RESUMEN FINAL
# ============================================================

resumen = pd.DataFrame(
    [
        {
            "z": z,
            "bits_estado": 25 * z,
            "rho_validado": is_permutation(
                create_labeled_state(z),
                rho(create_labeled_state(z)),
            ),
            "pi_validado": is_permutation(
                create_labeled_state(z),
                pi(create_labeled_state(z)),
            ),
            "rho_pi_validado": is_permutation(
                create_labeled_state(z),
                rho_pi(create_labeled_state(z)),
            ),
        }
        for z in (4, 8)
    ]
)

resumen

,z,bits_estado,rho_validado,pi_validado,rho_pi_validado
0,4,100,True,True,True
1,8,200,True,True,True


In [15]:
# ============================================================
# CONTROL FINAL
# ============================================================

assert resumen[
    [
        "rho_validado",
        "pi_validado",
        "rho_pi_validado",
    ]
].to_numpy().all()

print("=" * 70)
print("CAPAS RHO Y PI VALIDADAS")
print("=" * 70)
print("El proyecto está listo para implementar la capa theta.")
print("=" * 70)

CAPAS RHO Y PI VALIDADAS
El proyecto está listo para implementar la capa theta.
